## Practicals

In [5]:
# Demonstration: Transformer for NLP (Text Classification) - Fixed Version
# ------------------------------------------------------
# We'll build a Transformer-based text classifier without torchtext dependencies

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import numpy as np
import re
import random
from collections import Counter

# 1. Create Synthetic Movie Review Data (avoiding torchtext)
class SyntheticMovieDataset:
    def __init__(self):
        # Create synthetic movie reviews for demonstration
        positive_reviews = [
            "This movie is absolutely fantastic and amazing",
            "Brilliant acting and wonderful storyline throughout",
            "Excellent film with great performances by all actors",
            "Outstanding cinematography and superb direction",
            "Incredible movie with beautiful scenes and perfect music",
            "Amazing plot with excellent character development",
            "Wonderful film that exceeded all my expectations",
            "Fantastic acting and brilliant screenplay writing",
            "Superb movie with outstanding visual effects",
            "Excellent story with amazing performances",
        ] * 10  # Repeat to get 100 samples
        
        negative_reviews = [
            "This movie is terrible and completely boring",
            "Awful acting and horrible storyline throughout",
            "Poor film with bad performances by actors",
            "Terrible cinematography and awful direction",
            "Horrible movie with ugly scenes and terrible music",
            "Bad plot with poor character development",
            "Awful film that disappointed all expectations",
            "Terrible acting and poor screenplay writing",
            "Bad movie with horrible visual effects",
            "Poor story with terrible performances",
        ] * 10  # Repeat to get 100 samples
        
        # Combine and shuffle
        self.data = [(text, 1) for text in positive_reviews] + [(text, 0) for text in negative_reviews]
        random.shuffle(self.data)
    
    def get_train_data(self):
        return self.data[:160]  # 80% for training
    
    def get_test_data(self):
        return self.data[160:]  # 20% for testing

# 2. Simple Tokenizer and Vocabulary Builder
class SimpleTokenizer:
    def __init__(self):
        self.word_to_idx = {"<pad>": 0, "<unk>": 1}
        self.idx_to_word = {0: "<pad>", 1: "<unk>"}
        self.vocab_size = 2
    
    def tokenize(self, text):
        # Simple tokenization: lowercase, split by spaces, remove punctuation
        text = re.sub(r'[^\w\s]', '', text.lower())
        return text.split()
    
    def build_vocab(self, texts):
        word_counts = Counter()
        for text in texts:
            tokens = self.tokenize(text)
            word_counts.update(tokens)
        
        # Add most common words to vocabulary
        for word, count in word_counts.most_common(1000):  # Top 1000 words
            if word not in self.word_to_idx:
                self.word_to_idx[word] = self.vocab_size
                self.idx_to_word[self.vocab_size] = word
                self.vocab_size += 1
    
    def text_to_indices(self, text, max_length=50):
        tokens = self.tokenize(text)[:max_length]  # Truncate if too long
        indices = [self.word_to_idx.get(token, 1) for token in tokens]  # 1 = <unk>
        
        # Pad if too short
        while len(indices) < max_length:
            indices.append(0)  # 0 = <pad>
        
        return indices

# 3. PyTorch Dataset Class
class MovieDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=50):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        text, label = self.data[idx]
        indices = self.tokenizer.text_to_indices(text, self.max_length)
        return torch.tensor(indices, dtype=torch.long), torch.tensor(label, dtype=torch.long)

# 4. Transformer Model Definition
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, num_heads=4, num_layers=2, num_classes=2, max_seq_len=50):
        super().__init__()
        self.embed_dim = embed_dim
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_encoding = self._create_positional_encoding(max_seq_len, embed_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=num_heads, 
            dim_feedforward=128,
            dropout=0.1,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, num_classes)
        )

    def _create_positional_encoding(self, max_len, d_model):
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                           (-np.log(10000.0) / d_model))
        print(pe)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)  # Add batch dimension

    def forward(self, x):
        # x shape: (batch_size, seq_len)
        seq_len = x.size(1)
        
        # Create padding mask (True for padding tokens)
        padding_mask = (x == 0)
        
        # Embedding + positional encoding
        x = self.embedding(x) * np.sqrt(self.embed_dim)  # Scale embeddings
        x = x + self.pos_encoding[:, :seq_len, :].to(x.device)
        
        # Transformer encoding
        x = self.transformer(x, src_key_padding_mask=padding_mask)
        
        # Global average pooling (ignore padding tokens)
        mask = (~padding_mask).float().unsqueeze(-1)  # Convert to float and add feature dim
        x = (x * mask).sum(dim=1) / mask.sum(dim=1)  # Masked average
        
        # Classification
        output = self.classifier(x)
        return output

# 5. Data Preparation
print("Creating synthetic dataset...")
dataset_creator = SyntheticMovieDataset()
train_data = dataset_creator.get_train_data()
test_data = dataset_creator.get_test_data()

# Build vocabulary
tokenizer = SimpleTokenizer()
all_texts = [text for text, _ in train_data + test_data]
tokenizer.build_vocab(all_texts)
print(f"Vocabulary size: {tokenizer.vocab_size}")

# Create datasets and data loaders
train_dataset = MovieDataset(train_data, tokenizer)
test_dataset = MovieDataset(test_data, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# 6. Model Training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = TransformerClassifier(vocab_size=tokenizer.vocab_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("\nStarting training...")
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, (texts, labels) in enumerate(train_loader):
        texts, labels = texts.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(texts)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}: Loss = {avg_loss:.4f}, Accuracy = {accuracy:.2f}%")

# 7. Test Evaluation
print("\nEvaluating on test set...")
model.eval()
test_correct = 0
test_total = 0

with torch.no_grad():
    for texts, labels in test_loader:
        texts, labels = texts.to(device), labels.to(device)
        outputs = model(texts)
        _, predicted = torch.max(outputs.data, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

test_accuracy = 100 * test_correct / test_total
print(f"Test Accuracy: {test_accuracy:.2f}%")

# 8. Interactive Testing
def predict_sentiment(text):
    model.eval()
    with torch.no_grad():
        indices = tokenizer.text_to_indices(text)
        tensor = torch.tensor(indices).unsqueeze(0).to(device)
        output = model(tensor)
        probabilities = F.softmax(output, dim=1)
        predicted_class = torch.argmax(output, dim=1).item()
        confidence = probabilities[0][predicted_class].item()
        
        sentiment = "Positive" if predicted_class == 1 else "Negative"
        return sentiment, confidence

# Test with sample reviews
test_reviews = [
    "This movie is absolutely fantastic and amazing!",
    "Terrible film with awful acting and poor story.",
    "Great performances and excellent cinematography.",
    "Boring movie with bad direction and terrible script."
]

print("\n" + "="*50)
print("TESTING SAMPLE REVIEWS:")
print("="*50)

for review in test_reviews:
    sentiment, confidence = predict_sentiment(review)
    print(f"Review: '{review}'")
    print(f"Prediction: {sentiment} (Confidence: {confidence:.3f})")
    print("-" * 50)

print(f"\n✅ Transformer classifier training completed successfully!")
print(f"📊 Final test accuracy: {test_accuracy:.2f}%")
print(f"🔧 Model uses {sum(p.numel() for p in model.parameters()):,} parameters")

Creating synthetic dataset...
Vocabulary size: 52
Using device: cpu
tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

Starting training...
Epoch 1/5: Loss = 0.6108, Accuracy = 69.38%
Epoch 1/5: Loss = 0.6108, Accuracy = 69.38%
Epoch 2/5: Loss = 0.3615, Accuracy = 96.88%
Epoch 2/5: Loss = 0.3615, Accuracy = 96.88%
Epoch 3/5: Loss = 0.1198, Accuracy = 100.00%
Epoch 3/5: Loss = 0.1198, Accuracy = 100.00%
Epoch 4/5: Loss = 0.0216, Accuracy = 100.00%
Epoch 4/5: Loss = 0.0216, Accuracy = 100.00%
Epoch 5/5: Loss = 0.0081, Accuracy = 100.00%

Evaluating on test set...
Test Accuracy: 100.00%

TESTING SAMPLE REVIEWS:
Review: 'This movie is absolutely fantastic and amazing!'
Prediction: Positive (Confidence: 0.995)
--------------------------------------------------
Review: 'Terrible film with aw

In [9]:
import torch
import torch.nn as nn

layer = nn.Linear(in_features=4, out_features=2)
x = torch.tensor([[1.0, 2.0, 3.0, 4.0]])  # shape (1,4)
y = layer(x)
print(y.shape)

torch.Size([1, 2])
